# Quantum Simulator

In [1]:
import numpy as np
from functools import reduce

## Gates and Input vector

In [2]:
def Qubit1(val):
    A = np.zeros(2)
    A[val] = 1
    return A

def basis(string):
    input_states = []
    
    for char in string:
        if char == '0':
            input_states.append(Qubit1(0))
        elif char == '1':
            input_states.append(Qubit1(1))
        elif char == '+':
            input_states.append((Qubit1(0) + Qubit1(1)) / np.sqrt(2))
        elif char == '-':
            input_states.append((Qubit1(0) - Qubit1(1)) / np.sqrt(2))

    result = input_states[0]
    for state in input_states[1:]:
        result = np.kron(result, state)
        
    return result

In [3]:
X_mat = np.array([[0, 1], [1, 0]])
Y_mat = np.array([[0, -1j], [1j, 0]])
Z_mat = np.array([[1, 0], [0, -1]])
H_mat = 1 / np.sqrt(2) * np.array([[1, 1], [1, -1]])
T_mat = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])
SWAP_mat = np.array([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]])

def xgate(n_qubits):
    step = X_mat
    for _ in range(n_qubits - 1):
        step = np.kron(step, X_mat)
    return step


def ygate(n_qubits):
    step = Y_mat
    for _ in range(n_qubits - 1):
        step = np.kron(step, Y_mat)
    return step


def zgate(n_qubits):
    step = Z_mat
    for _ in range(n_qubits - 1):
        step = np.kron(step, Z_mat)
    return step


def Hgate(n_qubits):
    step = H_mat
    for _ in range(n_qubits - 1):
        step = np.kron(step, H_mat)
    return step


def Tgate(n_qubits):
    step = T_mat
    for _ in range(n_qubits - 1):
        step = np.kron(step, T_mat)
    return step


def R_k(k):
    """Single-qubit phase shift gate R_k."""
    phase = np.exp(2j * np.pi / (2**k))
    return np.array([[1, 0], [0, phase]], dtype=complex)

def CNOTgate(n_qubits, control, target):
    """
    Constructs an n-qubit CNOT matrix with specified control and target qubit indices.
    Indices are 0-based from left to right.
    """
    P0 = np.array([[1, 0], [0, 0]])
    P1 = np.array([[0, 0], [0, 1]])
    I  = np.eye(2)
    X  = np.array([[0, 1], [1, 0]])

    # Term 1: Control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(n_qubits)]
    
    # Term 2: Control is |1> -> apply X to target
    term1 = [X if i == target else (P1 if i == control else I) for i in range(n_qubits)]

    # Compute tensor products
    op0 = term0[0]
    op1 = term1[0]
    for gate0, gate1 in zip(term0[1:], term1[1:]):
        op0 = np.kron(op0, gate0)
        op1 = np.kron(op1, gate1)

    return op0 + op1

def CZgate(n_qubits, control, target):
    """
    Constructs an n-qubit CZ matrix with specified control and target qubit indices.
    Indices are 0-based from left to right.
    """
    P0 = np.array([[1, 0], [0, 0]])
    P1 = np.array([[0, 0], [0, 1]])
    I  = np.eye(2)
    Z  = zgate(1)

    # Term 1: Control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(n_qubits)]
    
    # Term 2: Control is |1> -> apply X to target
    term1 = [Z if i == target else (P1 if i == control else I) for i in range(n_qubits)]

    # Compute tensor products
    op0 = term0[0]
    op1 = term1[0]
    for gate0, gate1 in zip(term0[1:], term1[1:]):
        op0 = np.kron(op0, gate0)
        op1 = np.kron(op1, gate1)

    return op0 + op1

def CR_k(n_qubits, control, target, k):
    """Constructs an n-qubit Controlled-R_k gate matrix."""
    P0 = np.array([[1, 0], [0, 0]], dtype=complex)
    P1 = np.array([[0, 0], [0, 1]], dtype=complex)
    I = np.eye(2, dtype=complex)
    R = R_k(k)

    term0_list = [P0 if i == control else I for i in range(n_qubits)]
    term1_list = [R if i == target else (P1 if i == control else I) for i in range(n_qubits)]

    op0 = reduce(np.kron, term0_list)
    op1 = reduce(np.kron, term1_list)
    
    return op0 + op1

def swap_gate(n_qubits, q1, q2):
    """Constructs an n-qubit SWAP matrix between qubit q1 and q2."""
    cnot1 = CNOTgate(n_qubits, q1, q2)
    cnot2 = CNOTgate(n_qubits, q2, q1)
    cnot3 = CNOTgate(n_qubits, q1, q2)
    return cnot3 @ cnot2 @ cnot1

In [4]:
def apply_single_gate(gate_mat, n_qubits, target):
    I = np.eye(2, dtype=complex)
    """
    Applies a single-qubit gate matrix to 'target' qubit (0-indexed).
    Leaves all other qubits unchanged using Identity matrices.
    """
    ops = [gate_mat if i == target else I for i in range(n_qubits)]
    return reduce(np.kron, ops)

def xgatesingle(n_qubits, target):
    return apply_single_gate(X_mat, n_qubits, target)

def Hgatesingle(n_qubits, target):
    return apply_single_gate(H_mat, n_qubits, target)

def Tgatesingle(n_qubits, target):
    return apply_single_gate(T_mat, n_qubits, target)


In [5]:
def controlled_gate(gate_mat, n_qubits, control, target):
    """
    Constructs an n-qubit controlled gate for any 2x2 matrix U.
    """
    I = np.eye(2, dtype=complex)
    P0 = np.array([[1, 0], [0, 0]], dtype=complex)
    P1 = np.array([[0, 0], [0, 1]], dtype=complex)

    term0 = [P0 if i == control else I for i in range(n_qubits)]
    term1 = [gate_mat if i == target else (P1 if i == control else I) for i in range(n_qubits)]

    return reduce(np.kron, term0) + reduce(np.kron, term1)

In [6]:
Input = basis('000')
n_qubit = 3
step1 = Hgatesingle(n_qubit,0)
step2 = CR_k(n_qubit, 1,0,2) 
step3 = CR_k(n_qubit, 2,0,3)
step4 = Hgatesingle(n_qubit,1)
step5 = CR_k(n_qubit, 2,1,2)
step6 = Hgatesingle(n_qubit,2)
step7 = swap_gate(n_qubit, 0, 2)

full_circuit = step7 @ step6 @ step5 @ step4 @ step3 @ step2 @ step1
output = full_circuit @ Input
output

array([0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j,
       0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j, 0.35355339+0.j])

2
3
4


In [ ]:
def QFT(input):
    n_qubits=np.log2(len(input))
    step = input
    if n_qubits == 1:
        output = Hgate(n_qubits) * step
    else:
        for i in range(0,n_qubits):
            step = Hgatesingle(n_qubits,i) * step
            for j in range(2,n_qubits):
                step = CR_k(n_qubits, i,j-1,j) * step
    for i in range(n_qubits):
        step = swap_gate(n_qubits,i,n_qubits-i) * step
    return step

QFT(basis('000'))


TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [14]:
print(basis('0'),basis('00'),basis('000'),basis('0000'))

[1. 0.] [1. 0. 0. 0.] [1. 0. 0. 0. 0. 0. 0. 0.] [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:
xcheck0 = xgate(1) @ basis('0')
xcheck1 = xgate(1) @ basis('1')

ycheck0 = ygate(1) @ basis('0')
ycheck1 = ygate(1) @ basis('1')

zcheck0 = zgate(1) @ basis('0')
zcheck1 = zgate(1) @ basis('1')

Hcheck0 = Hgate(1) @ basis('0')
Hcheck1 = Hgate(1) @ basis('1')

Tcheck0 = Tgate(1) @ basis('0')
Tcheck1 = Tgate(1) @ basis('1')

CNOTcheck0 = CNOTgate(2, 0, 1) @ basis('10')
CNOTcheck1 = CNOTgate(2, 0, 1) @ basis('11')

CZcheck0 = CZgate(2, 0, 1) @ basis('10')
CZcheck1 = CZgate(2, 0, 1) @ basis('11')

display(xcheck0, xcheck1)
display(ycheck0, ycheck1)
display(zcheck0, zcheck1)
display(Hcheck0, Hcheck1)
display(Tcheck0, Tcheck1)
display(CNOTcheck0, CNOTcheck1)
display(CZcheck0, CZcheck1)

array([0., 1.])

array([1., 0.])

array([0.+0.j, 0.+1.j])

array([0.-1.j, 0.+0.j])

array([1., 0.])

...

array([0.        +0.j        , 0.70710678+0.70710678j])

array([0., 0., 0., 1.])

array([0., 0., 1., 0.])

array([0., 0., 1., 0.])

array([ 0.,  0.,  0., -1.])